In [0]:
%pip install faker

In [0]:
from faker import Faker
import pandas as pd
import random
from datetime import datetime, timedelta

fake = Faker("en_IN")

In [0]:
customers = []

customer_types = ["REGULAR", "PREMIUM", "VIP"]

for i in range(1, 501):

    customer_id = f"CUST{i:04d}"

    customer_name = fake.name()

    email = fake.email()

    registration_date = fake.date_between(start_date='-3y', end_date='today')

    customer_type = random.choice(customer_types)

    customers.append([
        customer_id,
        customer_name,
        email,
        registration_date,
        customer_type
    ])

customers_df = pd.DataFrame(
    customers,
    columns=[
        "customer_id",
        "customer_name",
        "email",
        "registration_date",
        "customer_type"
    ]
)

customers_df.head()

In [0]:
invalid_rows = random.sample(range(500), 10)

for i in invalid_rows:
    customers_df.loc[i, "email"] = customers_df.loc[i, "email"].replace("@", "")

In [0]:
customers_df.iloc[invalid_rows]

In [0]:
customers_df.shape

In [0]:
customers_df.to_csv(
    "/Volumes/workspace/default/assignment8/customers.csv",
    index=False
)

In [0]:
display(
    spark.read.csv(
        "/Volumes/workspace/default/assignment8/customers.csv",
        header=True,
        inferSchema=True
    )
)

In [0]:
categories = {
    "Electronics": ["Mobile", "Laptop", "Headphones", "Camera"],
    "Clothing": ["Shirt", "Jeans", "Jacket", "Shoes"],
    "Home": ["Chair", "Table", "Sofa", "Lamp"],
    "Books": ["Novel", "Education", "Comics", "Biography"]
}

products = []

for i in range(1, 501):

    product_id = f"PROD{i:04d}"

    category = random.choice(list(categories.keys()))

    subcategory = random.choice(categories[category])

    product_name = fake.word().capitalize() + " " + subcategory

    cost_price = random.randint(100, 50000)

    products.append([
        product_id,
        product_name,
        category,
        subcategory,
        cost_price
    ])

products_df = pd.DataFrame(
    products,
    columns=[
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "cost_price"
    ]
)

products_df.head()

In [0]:
rows = random.sample(range(500), 20)

for i in rows:

    if i % 2 == 0:
        products_df.loc[i, "product_name"] = "   " + products_df.loc[i, "product_name"] + "   "
    else:
        products_df.loc[i, "product_name"] = products_df.loc[i, "product_name"].upper()

In [0]:
products_df.to_csv(
    "/Volumes/workspace/default/assignment8/products.csv",
    index=False
)

In [0]:
display(
    spark.read.csv(
        "/Volumes/workspace/default/assignment8/products.csv",
        header=True,
        inferSchema=True
    )
)

In [0]:
statuses = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]

regions = ["NORTH", "SOUTH", "EAST", "WEST"]

orders = []

for i in range(1,701):

    order_id = f"ORD{i:05d}"

    customer_id = f"CUST{random.randint(1,500):04d}"

    order_date = fake.date_time_between(
        start_date='-2y',
        end_date='now'
    )

    status = random.choice(statuses)

    region = random.choice(regions)

    orders.append([
        order_id,
        customer_id,
        order_date,
        status,
        region
    ])

orders_df = pd.DataFrame(
    orders,
    columns=[
        "order_id",
        "customer_id",
        "order_date",
        "status",
        "region_code"
    ]
)

orders_df.head()

In [0]:
null_rows = random.sample(range(700),35)

orders_df.loc[null_rows,"customer_id"] = None

In [0]:
wrong_dates = random.sample(range(700),20)

for i in wrong_dates:

    orders_df.loc[i,"order_date"] = pd.to_datetime(
        orders_df.loc[i,"order_date"]
    ).strftime("%d-%m-%Y")

In [0]:
orders_df.to_csv(
    "/Volumes/workspace/default/assignment8/orders.csv",
    index=False
)

In [0]:
display(
spark.read.csv(
"/Volumes/workspace/default/assignment8/orders.csv",
header=True,
inferSchema=True
)
)

In [0]:
order_items = []

for i in range(1,2501):

    item_id = f"ITEM{i:05d}"

    order_id = f"ORD{random.randint(1,700):05d}"

    product_id = f"PROD{random.randint(1,500):04d}"

    quantity = random.randint(1,5)

    unit_price = random.randint(200,60000)

    discount_percent = random.randint(0,100)

    order_items.append([
        item_id,
        order_id,
        product_id,
        quantity,
        unit_price,
        discount_percent
    ])

order_items_df = pd.DataFrame(
    order_items,
    columns=[
        "item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount_percent"
    ]
)

order_items_df.head()

In [0]:
negative_rows = random.sample(range(2500),75)

for i in negative_rows:

    order_items_df.loc[i,"quantity"] = -random.randint(1,5)

In [0]:
order_items_df.to_csv(
    "/Volumes/workspace/default/assignment8/order_items.csv",
    index=False
)

In [0]:
display(
spark.read.csv(
"/Volumes/workspace/default/assignment8/order_items.csv",
header=True,
inferSchema=True
)
)

# E-Commerce Order Analytics System

## Phase 2 - Data Cleaning

In this phase, we clean the generated datasets by:
- Fixing incorrect date formats
- Handling missing customer IDs
- Normalizing product names
- Validating customer emails
- Checking referential integrity

## Step 1: Load Generated CSV Files

In [0]:
orders = pd.read_csv("/Volumes/workspace/default/assignment8/orders.csv")

customers = pd.read_csv("/Volumes/workspace/default/assignment8/customers.csv")

products = pd.read_csv("/Volumes/workspace/default/assignment8/products.csv")

order_items = pd.read_csv("/Volumes/workspace/default/assignment8/order_items.csv")

## Step 2: Check Dataset Information

In [0]:
print("Orders")
orders.info()

print("\nCustomers")
customers.info()

print("\nProducts")
products.info()

print("\nOrder Items")
order_items.info()

## Step 3: Clean Orders Dataset

In [0]:
def clean_orders(df):

    df = df.copy()

    # Convert dates
    df["order_date"] = pd.to_datetime(
        df["order_date"],
        errors="coerce",
        dayfirst=True
    )

    # Replace missing customer IDs
    df["customer_id"] = df["customer_id"].fillna("UNKNOWN")

    return df

In [0]:
orders_clean = clean_orders(orders)

orders_clean.head()

## Step 4: Verify Cleaned Orders

In [0]:
orders_clean.info()

In [0]:
orders_clean.to_csv(
    "/Volumes/workspace/default/assignment8/orders_clean.csv",
    index=False
)

## Step 5: Clean Products Dataset

In [0]:
def clean_products(df):

    df = df.copy()

    df["product_name"] = (
        df["product_name"]
        .str.strip()
        .str.title()
    )

    return df

In [0]:
products_clean = clean_products(products)

products_clean.head(20)

## Step 6: Verify Product Cleaning

In [0]:
products_clean.info()

In [0]:
products_clean.to_csv(
    "/Volumes/workspace/default/assignment8/products_clean.csv",
    index=False
)

## Step 7: Validate Customer Emails

In [0]:
def validate_emails(df):

    invalid = df[
        ~df["email"].str.contains("@", na=False)
    ]

    return invalid

In [0]:
invalid_emails = validate_emails(customers)

invalid_emails

## Step 8: Check Referential Integrity

In [0]:
def check_referential_integrity(order_items_df, orders_df):

    invalid_orders = order_items_df[
        ~order_items_df["order_id"].isin(orders_df["order_id"])
    ]

    return invalid_orders

In [0]:
invalid_order_items = check_referential_integrity(
    order_items,
    orders
)

invalid_order_items

## Step 9: Data Quality Summary

In [0]:
print("Orders:", len(orders))
print("Customers:", len(customers))
print("Products:", len(products))
print("Order Items:", len(order_items))

print("\nInvalid Emails:", len(invalid_emails))

print("Missing Customer IDs:",
      orders["customer_id"].isna().sum())

print("Negative Quantity:",
      (order_items["quantity"] < 0).sum())

print("Broken Order References:",
      len(invalid_order_items))

In [0]:
customers.to_csv(
    "/Volumes/workspace/default/assignment8/customers_clean.csv",
    index=False
)

products_clean.to_csv(
    "/Volumes/workspace/default/assignment8/products_clean.csv",
    index=False
)

orders_clean.to_csv(
    "/Volumes/workspace/default/assignment8/orders_clean.csv",
    index=False
)

order_items.to_csv(
    "/Volumes/workspace/default/assignment8/order_items_clean.csv",
    index=False
)

print("All cleaned files saved successfully!")

# Phase 3 - SQL Analysis

## Step 1: Load Cleaned CSV Files into Spark DataFrames

In [0]:
orders_sql = spark.read.csv(
    "/Volumes/workspace/default/assignment8/orders_clean.csv",
    header=True,
    inferSchema=True
)

customers_sql = spark.read.csv(
    "/Volumes/workspace/default/assignment8/customers_clean.csv",
    header=True,
    inferSchema=True
)

products_sql = spark.read.csv(
    "/Volumes/workspace/default/assignment8/products_clean.csv",
    header=True,
    inferSchema=True
)

order_items_sql = spark.read.csv(
    "/Volumes/workspace/default/assignment8/order_items_clean.csv",
    header=True,
    inferSchema=True
)

## Step 2: Create Temporary SQL Views

In [0]:
orders_sql.createOrReplaceTempView("orders")

customers_sql.createOrReplaceTempView("customers")

products_sql.createOrReplaceTempView("products")

order_items_sql.createOrReplaceTempView("order_items")

## Step 3: Verify the Tables

In [0]:
spark.sql("SHOW TABLES").show()

# SQL Query 1

## Total Revenue per Category

In [0]:
query1 = """
SELECT
    p.category,

    SUM(
        oi.quantity *
        oi.unit_price *
        (1 - oi.discount_percent/100)
    ) AS total_revenue

FROM order_items oi

JOIN products p
ON oi.product_id = p.product_id

GROUP BY p.category

ORDER BY total_revenue DESC
"""

In [0]:
spark.sql(query1).show()

# SQL Query 2

## Top 10 Customers by Total Order Value

In [0]:
query2 = """

SELECT

    c.customer_id,

    c.customer_name,

    ROUND(

        SUM(

            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent/100)

        ),2

    ) AS total_order_value

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY

    c.customer_id,

    c.customer_name

ORDER BY total_order_value DESC

LIMIT 10

"""

In [0]:
display(
    spark.sql(query2)
)

# SQL Query 3

## Month-wise Order Count for Last 12 Months

In [0]:
query3 = """

SELECT

DATE_FORMAT(order_date,'yyyy-MM') AS month,

COUNT(order_id) AS total_orders

FROM orders

WHERE order_date >= add_months(current_date(),-12)

GROUP BY DATE_FORMAT(order_date,'yyyy-MM')

ORDER BY month

"""

In [0]:
display(
    spark.sql(query3)
)

# SQL Query 4

## Find Customers Who Placed Orders but Never Had Any Item Delivered

In [0]:
query4 = """

SELECT DISTINCT

    c.customer_id,
    c.customer_name

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

WHERE c.customer_id NOT IN (

    SELECT DISTINCT customer_id

    FROM orders

    WHERE status='DELIVERED'

)

"""

In [0]:
display(
    spark.sql(query4)
)

# SQL Query 5

## Products Ordered but Returned More Than Purchased

In [0]:
query5 = """

SELECT

    p.product_id,

    p.product_name,

    SUM(
        CASE
            WHEN oi.quantity>0 THEN oi.quantity
            ELSE 0
        END
    ) AS purchased,

    ABS(

        SUM(

            CASE
                WHEN oi.quantity<0 THEN oi.quantity
                ELSE 0
            END

        )

    ) AS returned

FROM products p

JOIN order_items oi

ON p.product_id=oi.product_id

GROUP BY

p.product_id,
p.product_name

HAVING returned > purchased

"""

In [0]:
display(
    spark.sql(query5)
)

# SQL Query 6

## Return Rate Per Category

In [0]:
query6 = """

SELECT

p.category,

ROUND(

ABS(

SUM(

CASE

WHEN oi.quantity<0

THEN oi.quantity

ELSE 0

END

)

)

/

SUM(

ABS(oi.quantity)

)

*100,

2

)

AS return_rate

FROM products p

JOIN order_items oi

ON p.product_id=oi.product_id

GROUP BY p.category

ORDER BY return_rate DESC

"""

In [0]:
display(
    spark.sql(query6)
)

# SQL Query 7

## Running Total of Revenue per Region

In [0]:
query7 = """

WITH daily_revenue AS (

SELECT

o.region_code,

DATE(o.order_date) AS order_date,

SUM(

oi.quantity *
oi.unit_price *
(1-oi.discount_percent/100)

) AS daily_revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY

o.region_code,

DATE(o.order_date)

)

SELECT

region_code,

order_date,

ROUND(daily_revenue,2) AS daily_revenue,

ROUND(

SUM(daily_revenue)

OVER(

PARTITION BY region_code

ORDER BY order_date

),

2

)

AS running_total

FROM daily_revenue

ORDER BY

region_code,

order_date

"""

In [0]:
display(
spark.sql(query7)
)

# SQL Query 8

## Rank Products by Revenue using DENSE_RANK

In [0]:
query8 = """

SELECT

category,

product_name,

total_revenue,

DENSE_RANK()

OVER(

PARTITION BY category

ORDER BY total_revenue DESC

)

AS rank_in_category

FROM(

SELECT

p.category,

p.product_name,

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100)

)

AS total_revenue

FROM products p

JOIN order_items oi

ON p.product_id=oi.product_id

GROUP BY

p.category,

p.product_name

)t

ORDER BY

category,

rank_in_category

"""

In [0]:
display(
spark.sql(query8)
)

# SQL Query 9

## Days Between Consecutive Orders (LAG)

In [0]:
query9 = """

SELECT

customer_id,

order_date,

previous_order_date,

DATEDIFF(
order_date,
previous_order_date
)

AS days_gap

FROM(

SELECT

customer_id,

order_date,

LAG(order_date)

OVER(

PARTITION BY customer_id

ORDER BY order_date

)

AS previous_order_date

FROM orders

)t

ORDER BY

customer_id,

order_date

"""

In [0]:
display(
spark.sql(query9)
)

# SQL Query 10

## Monthly Revenue Category using Multiple CTEs

In [0]:
query10 = """

WITH monthly_revenue AS(

SELECT

o.customer_id,

DATE_FORMAT(
o.order_date,
'yyyy-MM'
)

AS month,

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100)

)

AS revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY

o.customer_id,

DATE_FORMAT(
o.order_date,
'yyyy-MM'
)

),

customer_category AS(

SELECT

customer_id,

month,

revenue,

CASE

WHEN revenue>10000 THEN 'High'

WHEN revenue BETWEEN 5000 AND 10000 THEN 'Medium'

ELSE 'Low'

END

AS category

FROM monthly_revenue

)

SELECT

month,

category,

COUNT(customer_id)

AS total_customers

FROM customer_category

GROUP BY

month,

category

ORDER BY

month,

category

"""

In [0]:
display(
spark.sql(query10)
)

# SQL Query 11

## Customer Segmentation using NTILE

In [0]:
query11 = """

WITH customer_revenue AS (

SELECT

o.customer_id,

SUM(

oi.quantity *
oi.unit_price *
(1 - oi.discount_percent/100)

) AS total_value

FROM orders o

JOIN order_items oi

ON o.order_id = oi.order_id

GROUP BY o.customer_id

)

SELECT

customer_id,

ROUND(total_value,2) AS total_value,

NTILE(4) OVER(

ORDER BY total_value DESC

) AS quartile,

CASE

WHEN NTILE(4) OVER(ORDER BY total_value DESC)=1 THEN 'Platinum'
WHEN NTILE(4) OVER(ORDER BY total_value DESC)=2 THEN 'Gold'
WHEN NTILE(4) OVER(ORDER BY total_value DESC)=3 THEN 'Silver'
ELSE 'Bronze'

END AS quartile_label

FROM customer_revenue

"""

In [0]:
display(
spark.sql(query11)
)

# SQL Query 12

## Year-over-Year Revenue Comparison

In [0]:
query12 = """

WITH monthly_sales AS(

SELECT

YEAR(o.order_date) AS year,

MONTH(o.order_date) AS month,

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100)

) AS revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY

YEAR(o.order_date),

MONTH(o.order_date)

)

SELECT

year,

month,

ROUND(revenue,2) AS revenue,

LAG(revenue)

OVER(

PARTITION BY month

ORDER BY year

)

AS prev_year_revenue,

ROUND(

(

revenue -

LAG(revenue)

OVER(

PARTITION BY month

ORDER BY year

)

)

/

LAG(revenue)

OVER(

PARTITION BY month

ORDER BY year

)

*100,

2

)

AS yoy_growth_percent

FROM monthly_sales

ORDER BY

year,

month

"""

In [0]:
display(
spark.sql(query12)
)

# SQL Query 13

## First Purchased Category vs Latest Purchased Category

In [0]:
query13 = """

WITH customer_categories AS(

SELECT

o.customer_id,

p.category,

o.order_date,

ROW_NUMBER()

OVER(

PARTITION BY o.customer_id

ORDER BY o.order_date

) AS first_order,

ROW_NUMBER()

OVER(

PARTITION BY o.customer_id

ORDER BY o.order_date DESC

) AS last_order

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

JOIN products p

ON oi.product_id=p.product_id

)

SELECT

f.customer_id,

f.category AS first_category,

l.category AS last_category,

CASE

WHEN f.category=l.category

THEN 'No'

ELSE 'Yes'

END

AS category_shift

FROM customer_categories f

JOIN customer_categories l

ON f.customer_id=l.customer_id

WHERE

f.first_order=1

AND

l.last_order=1

"""

In [0]:
display(
spark.sql(query13)
)

# SQL Query 14

## Cumulative Revenue Distribution

In [0]:
query14 = """

WITH customer_revenue AS(

SELECT

o.customer_id,

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100)

)

AS revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY o.customer_id

)

SELECT

customer_id,

ROUND(revenue,2) AS revenue,

ROUND(

SUM(revenue)

OVER(

ORDER BY revenue DESC

),

2

)

AS cumulative_revenue,

ROUND(

SUM(revenue)

OVER(

ORDER BY revenue DESC

)

/

SUM(revenue)

OVER()

*100,

2

)

AS cumulative_percent

FROM customer_revenue

"""

In [0]:
display(
spark.sql(query14)
)

# SQL Query 15

## Customer Cohort Analysis

In [0]:
query15 = """

SELECT

DATE_FORMAT(
c.registration_date,
'yyyy-MM'
)

AS cohort,

COUNT(DISTINCT c.customer_id)

AS total_customers,

COUNT(DISTINCT o.customer_id)

AS ordered_customers

FROM customers c

LEFT JOIN orders o

ON c.customer_id=o.customer_id

GROUP BY

DATE_FORMAT(
c.registration_date,
'yyyy-MM'
)

ORDER BY cohort

"""

In [0]:
display(
spark.sql(query15)
)

# SQL Query 16

## Products Frequently Bought Together

In [0]:
query16 = """

SELECT

a.product_id AS product_a,

b.product_id AS product_b,

COUNT(*) AS times_bought_together

FROM order_items a

JOIN order_items b

ON a.order_id=b.order_id

AND a.product_id<b.product_id

GROUP BY

a.product_id,

b.product_id

ORDER BY

times_bought_together DESC

"""

In [0]:
display(
spark.sql(query16)
)

# Phase 4 - Python + SQL Integration

This section generates summary reports based on a selected report type and date range.

## Step 1: Define Report Parameters

In [0]:
report_type = "Monthly"

start_date = "2025-01-01"

end_date = "2026-12-31"

## Step 2: Generate Summary Report

In [0]:
summary_query = f"""

SELECT

COUNT(DISTINCT o.order_id) AS total_orders,

ROUND(

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100)

),2

) AS total_revenue,

COUNT(DISTINCT o.customer_id) AS unique_customers

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

WHERE

o.order_date

BETWEEN

'{start_date}'

AND

'{end_date}'

"""

In [0]:
display(
spark.sql(summary_query)
)

## Step 3: Top 3 Products

In [0]:
top_product_query = f"""

SELECT

p.product_name,

SUM(

oi.quantity

)

AS total_quantity

FROM products p

JOIN order_items oi

ON p.product_id=oi.product_id

JOIN orders o

ON oi.order_id=o.order_id

WHERE

o.order_date

BETWEEN

'{start_date}'

AND

'{end_date}'

GROUP BY

p.product_name

ORDER BY

total_quantity DESC

LIMIT 3

"""

In [0]:
display(
spark.sql(top_product_query)
)

## Step 4: Previous Period Comparison

In [0]:
comparison_query = """

WITH yearly AS(

SELECT

YEAR(order_date) AS year,

COUNT(order_id) AS total_orders

FROM orders

GROUP BY YEAR(order_date)

)

SELECT

year,

total_orders,

LAG(total_orders)

OVER(

ORDER BY year

)

AS previous_year_orders

FROM yearly

"""

In [0]:
display(
spark.sql(comparison_query)
)

# Phase 5 - Edge Case Handling

In [0]:
print(

"Broken Order References :",

len(invalid_order_items)

)

## Test Case 2: Discount Greater Than 100%

In [0]:
discount_error = order_items[
order_items["discount_percent"]>100
]

discount_error

## Test Case 3: Quantity Equal to Zero

In [0]:
zero_quantity = order_items[
order_items["quantity"]==0
]

zero_quantity